In [ ]:
import pandas as pd

json_file_path = "/Users/liyong.1024/Documents/protect/redline_platform/redline_platform_sample_251112_v2.xlsx"
output_file_path = json_file_path.replace(".json", ".xlsx")
df = pd.read_json(json_file_path)
pos_data = df[(df['label'] <= 2)]
pos_data.to_excel(output_file_path)



In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
from tqdm import tqdm
import os
from PIL import Image
import io
import warnings
ARROW_AVAILABLE = True
# pa.PyExtensionType.set_auto_load(True)
json_file_path = "/Users/liyong.1024/Documents/protect/redline_vl/redline_vl_attr_sample_251107_train_v2_vlsft_rm01_p3n1.json"
df = pd.read_json(json_file_path)

def is_valid_image(image_path):
    try:
        with Image.open(image_path) as img:
            img.verify()  # 验证图像完整性
        return True
    except Exception as e:
        warnings.warn(f"Invalid image {image_path}: {str(e)}")
        return False

# 检查DataFrame的基本信息
print(f"DataFrame shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# 检查是否存在images字段
if 'images' not in df.columns:
    print("Warning: 'images' column not found in DataFrame!")
    # 尝试查找可能的图片字段
    possible_image_cols = [col for col in df.columns if 'image' in col.lower() or 'img' in col.lower()]
    print(f"Possible image columns: {possible_image_cols}")
else:
    print(f"'images' column found. Data type: {df['images'].dtype}")
    
    # 检查images字段的前几个样本
    print("\nFirst few 'images' field samples:")
    for i in range(min(5, len(df))):
        sample = df.iloc[i]['images']
        print(f"Sample {i}: {type(sample)} - {sample}")

    # 提取所有图片地址
    all_image_paths = []
    
    # 遍历DataFrame中的每一行
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extracting image paths"):
        images_field = row['images']
        
        # 处理不同类型的images字段
        if isinstance(images_field, list):
            # 如果是列表，直接扩展
            all_image_paths.extend(images_field)
        elif isinstance(images_field, str):
            # 如果是字符串，尝试解析为JSON列表
            try:
                images_list = json.loads(images_field)
                if isinstance(images_list, list):
                    all_image_paths.extend(images_list)
                else:
                    print(f"Warning: Row {idx} - 'images' field is string but not a valid JSON list")
            except json.JSONDecodeError:
                print(f"Warning: Row {idx} - 'images' field is string but cannot be parsed as JSON")
        elif pd.isna(images_field) or images_field is None:
            print(f"Warning: Row {idx} - 'images' field is empty/None")
        else:
            print(f"Warning: Row {idx} - 'images' field has unexpected type: {type(images_field)}")

    # 去重
    unique_image_paths = list(set(all_image_paths))
    
    print(f"\nTotal image paths extracted: {len(all_image_paths)}")
    print(f"Unique image paths: {len(unique_image_paths)}")
    print(f"First 10 unique image paths:")
    for i, path in enumerate(unique_image_paths[:10]):
        print(f"{i+1}: {path}")

    # 可选：保存图片路径到文件
    output_dir = os.path.dirname(json_file_path)
    image_paths_file = os.path.join(output_dir, "extracted_image_paths.txt")
    
    a = unique_image_paths
    unique_image_paths = []
    with open(image_paths_file, 'w') as f:
        for path in a:
            unique_image_paths.append(f"/Users/liyong.1024/Workspace/xz-shield/{path}")
            f.write(f"/Users/liyong.1024/Workspace/xz-shield/{path}\n")
    
    print(f"\nImage paths saved to: {image_paths_file}")
    x = unique_image_paths[0]
    print(f"{len(unique_image_paths)}\n{unique_image_paths[0]}\n{os.path.exists(x)}\n{is_valid_image(x)}")

    # 可选：创建包含所有图片信息的新DataFrame
    image_df = pd.DataFrame({
        'image_path': unique_image_paths,
        'file_name': [os.path.basename(path) for path in unique_image_paths],
        'file_dir': [os.path.dirname(path) for path in unique_image_paths]
    })

        
    # 检查文件是否存在
    image_df['file_exists'] = image_df['image_path'].apply(
        lambda x: is_valid_image(x) if isinstance(x, str) else False
    )
    
    print(f"\nImage DataFrame shape: {image_df.shape}")
    print(f"Existing files: {image_df['file_exists'].sum()}/{len(image_df)}")
    
    # 保存image_df到parquet文件（如果需要）
    # parquet_file = os.path.join(output_dir, "image_metadata.parquet")
    # if ARROW_AVAILABLE:
    #     table = pa.Table.from_pandas(image_df)
    #     pq.write_table(table, parquet_file)
    #     print(f"Image metadata saved to Parquet: {parquet_file}")




In [17]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
ARROW_AVAILABLE = True
# pa.PyExtensionType.set_auto_load(True)
json_file_path = "/Users/liyong.1024/Documents/protect/redline_platform/redline_platform_sample_251112_train_v2_vlsft_rq02.json"
output_file_path = "/Users/liyong.1024/Documents/protect/redline_platform/redline_platform_sample_251112_train_v2_p2n1_vlsft_rq02.json"
df = pd.read_json(json_file_path)

pos_data = df[(df['output'] == '是')]
neg_data = df[(df['output'] == '否')]
df = pd.concat([pos_data]*2 + [neg_data], ignore_index=True)

df.to_json(output_file_path, orient='records', date_format='iso', force_ascii=False, indent=0,lines=True)




In [18]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
ARROW_AVAILABLE = True
# pa.PyExtensionType.set_auto_load(True)
json_file_path = "/Users/liyong.1024/Documents/protect/redline_platform/redline_platform_sample_251112_val_v2_vlsft_rq02.json"
pos_file_path = "/Users/liyong.1024/Documents/protect/redline_platform/redline_platform_sample_251112_val_v2_vlsft_rq02_pos.json"
neg_file_path = "/Users/liyong.1024/Documents/protect/redline_platform/redline_platform_sample_251112_val_v2_vlsft_rq02_neg.json"
df = pd.read_json(json_file_path)

pos_data = df[(df['output'] == '是')]
neg_data = df[(df['output'] == '否')]


pos_data.to_json(pos_file_path, orient='records', date_format='iso', force_ascii=False, indent=2)
neg_data.to_json(neg_file_path, orient='records', date_format='iso', force_ascii=False, indent=2)




In [5]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
ARROW_AVAILABLE = True

def split_train_val(file_path, version="v1"):
    train_file_path = file_path.replace(".xlsx", f"_train_{version}.xlsx")
    val_file_path = file_path.replace(".xlsx", f"_val_{version}.xlsx")
    df = pd.read_excel(file_path)
    # 设置随机种子
    np.random.seed(42)
    df_shuffled = df.sample(frac=1).reset_index(drop=True)
    train_num = int(len(df_shuffled) * 0.9)
    df_train = df_shuffled[:train_num]
    df_val = df_shuffled[train_num:]

    df_train.to_excel(train_file_path)
    df_val.to_excel(val_file_path)
    return [train_file_path, val_file_path]

def split_pos_neg(file_path):
    pos_file_path = file_path.replace(".xlsx", f"_pos.xlsx")
    neg_file_path = file_path.replace(".xlsx", f"_neg.xlsx")
    df = pd.read_excel(file_path)

    pos_data = df[(df['redline_label'] == 1)]
    neg_data = df[(df['redline_label'] == 0)]
    pos_data.to_excel(pos_file_path)
    neg_data.to_excel(neg_file_path)


In [6]:
file_path = "/Users/liyong.1024/Documents/protect/redline_platform/redline_platform_sample_251112_v2_val_v3.xlsx"
split_pos_neg(file_path)

# output_files = split_train_val(file_path, version="v1")

# for f in output_files:
#     split_pos_neg(f)
